# Model 3



Using $\text{Model}_{\text{pref}}$ as the base model, include one additional
alternative attribute and one interaction of a socioeconomic characteristic
with either the ASCs or one of the attributes. Present both the
specification and the estimation results (as defined previously).

# Importing packages

We first import some useful Python packages. pandas is an open source data analysis and manipulation tool built in the Python programming language. Biogeme is a Python package for estimating discrete choice models to explain and predict how people choose among alternatives.

In [1]:
import pandas as pd
import biogeme.database as db
import biogeme.biogeme as bio
from tqdm.notebook import tqdm
from biogeme.expressions import Beta, Variable, log, exp
from biogeme import models
from biogeme.results_processing import EstimationResults
from biogeme.results_processing import get_pandas_estimated_parameters, html_output, calc_p_value
from scipy.stats import chi2, norm

g++ not available, if using conda: `conda install gxx`
c:\Users\lisav\Desktop\ENTPE\3A\Data science\modelling\.venv\Lib\site-packages\tqdm_joblib\__init__.py:4: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


# Reading and preparing the data

We first import the data into Pandas. We read the data from a text file, where the data are separated by tabs. 

In [2]:
df = pd.read_csv('lpmc19.dat', sep='\t')
df

,trip_id,household_id,person_n,trip_n,travel_mode,purpose,fueltype,faretype,bus_scale,survey_year,...,dur_pt_access,dur_pt_rail,dur_pt_bus,dur_pt_int,pt_interchanges,dur_driving,cost_transit,cost_driving_fuel,cost_driving_ccharge,driving_traffic_percent
0,5,0,1,3,4,3,1,4,1.0,1,...,0.205556,0.000000,0.221111,0.000000,0,0.222222,1.5,0.77,0.0,0.097500
1,16,2,1,2,4,3,1,1,1.0,1,...,0.098611,0.000000,0.058333,0.000000,0,0.058611,1.5,0.16,0.0,0.123223
2,26,7,0,2,1,3,2,1,1.0,1,...,0.068333,0.000000,0.033889,0.000000,0,0.055278,1.5,0.10,0.0,0.105528
3,27,7,1,0,4,4,2,1,1.0,1,...,0.193889,0.516667,0.000000,0.100000,1,0.340833,1.5,2.04,0.0,0.280359
4,52,12,1,2,4,5,2,1,1.0,1,...,0.062500,0.000000,0.491944,0.094722,1,0.355556,3.0,1.19,0.0,0.249219
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,81040,17604,2,0,3,1,1,2,0.0,3,...,0.264444,0.000000,0.353333,0.176667,1,0.288889,0.0,0.85,0.0,0.175000
4996,81059,17606,1,2,3,3,6,1,1.0,3,...,0.100278,0.000000,0.360278,0.000000,0,0.281667,1.5,0.63,0.0,0.384615
4997,81062,17607,0,4,3,5,6,1,1.0,3,...,0.069444,0.000000,0.166389,0.000000,0,0.146111,1.5,0.28,0.0,0.653992
4998,81071,17610,0,0,4,3,2,1,1.0,3,...,0.206111,0.000000,0.400000,0.000000,0,0.259167,1.5,0.88,0.0,0.233655


In [3]:
df['age'].dropna()


0       54
1       41
2       49
3       51
4       56
        ..
4995    17
4996    33
4997    38
4998    33
4999    40
Name: age, Length: 5000, dtype: int64

We then import this database into Biogeme.

In [4]:
database = db.Database('london', df)

We identify the columns that will be used as variables in our model. 

In [5]:
trip_id = Variable('trip_id')
travel_mode = Variable('travel_mode')
purpose = Variable('purpose')
distance = Variable('distance')
dur_walking = Variable('dur_walking')
dur_cycling = Variable('dur_cycling')
dur_pt_access = Variable('dur_pt_access')
dur_pt_rail  = Variable('dur_pt_rail')
dur_pt_bus  = Variable('dur_pt_bus')
dur_pt_int = Variable('dur_pt_int')
dur_driving = Variable('dur_driving')
cost_transit = Variable('cost_transit')
cost_driving_fuel = Variable('cost_driving_fuel')
cost_driving_ccharge = Variable('cost_driving_ccharge')
driving_traffic_percent = Variable('driving_traffic_percent')
age = Variable('age')

In [6]:
database.num_rows()

5000

We define the total travel time by public transportation

In [7]:
dur_pt_total = dur_pt_access + dur_pt_rail + dur_pt_bus + dur_pt_int
cost_car = cost_driving_fuel + cost_driving_ccharge

We convert the Guilders into Euros.

In [8]:
POUNDS_TO_EUROS = 1.16
cost_car_euro = cost_car * POUNDS_TO_EUROS
cost_pt_euro = cost_transit * POUNDS_TO_EUROS

# Model specification

We can now specify a model that contains 4 ASC

In [9]:
asc_car = Beta('asc_car', 0, None, None, 0)
asc_walk = Beta('asc_walk', 0, None, None, 0)
asc_bike = Beta('asc_bike', 0, None, None, 0)
asc_pt = Beta('asc_pt', 0, None, None, 0)
beta_cost = Beta('beta_cost_car', 0, None, None, 0)
beta_age = Beta('beta_age', 0, None, None, 0)
beta_time_car = Beta('beta_time_car', 0, None, None, 0)
beta_time_walk = Beta('beta_time_walk', 0, None, None, 0)
beta_time_bike = Beta('beta_time_bike', 0, None, None, 0)
beta_time_pt = Beta('beta_time_pt', 0, None, None, 0)
beta_driving_traffic_percent = Beta('beta_driving_traffic_percent', 0, None, None, 0)

We  write the utility functions:
**We choose walking as the reference**

In [10]:
v_car = asc_car + beta_age * age + beta_cost * cost_car_euro + beta_time_car * log(dur_driving) + beta_driving_traffic_percent * driving_traffic_percent
v_walk = 0 + beta_time_walk * log(dur_walking)
v_bike = asc_bike + beta_age * age + beta_time_bike * log(dur_cycling)
v_pt = asc_pt + beta_age * age + beta_cost * cost_pt_euro + beta_time_pt * log(dur_pt_total)

And we write the choice model:

In [11]:
V = {1: v_walk, 2: v_bike, 3: v_pt, 4: v_car}

In [12]:
logprob = models.loglogit(V, None, travel_mode)

We initialize Biogeme with this expression and the database. We call the model 'binary_netherlands'.

In [13]:
biogeme = bio.BIOGEME(database, logprob)
biogeme.model_name = 'model_3'

# Model estimation

In [14]:
results_model_3 = biogeme.estimate()

We print some general statistics: 

In [15]:
print(results_model_3.print_general_statistics())

Number of estimated parameters             10
Sample size                                5000
Excluded observations                      0
Init log likelihood                        -6931.472
Final log likelihood                       -4121.26
Likelihood ratio test for the init. model  5620.423
Rho-square for the init. model             0.405
Rho-square-bar for the init. model         0.404
Akaike Information Criterion               8262.52
Bayesian Information Criterion             8327.692
Final gradient norm                        1.7638E-02
Bootstrapping time                         None


We print the estimation results

In [16]:
boxcox_estimates = get_pandas_estimated_parameters(estimation_results=results_model_3)
boxcox_estimates

{'Estimated parameters':                            Name     Value  Robust std err.  Robust t-stat.  \
 0                beta_time_walk -4.191312         0.127085      -32.980393   
 1                      asc_bike -2.575308         0.240312      -10.716541   
 2                      beta_age  0.011037         0.002456        4.494215   
 3                beta_time_bike -2.113964         0.130568      -16.190541   
 4                        asc_pt  1.403750         0.157965        8.886445   
 5                 beta_cost_car -0.122815         0.014454       -8.497253   
 6                  beta_time_pt -1.676739         0.117366      -14.286451   
 7                       asc_car  1.740407         0.242987        7.162548   
 8                 beta_time_car -1.542057         0.110661      -13.934966   
 9  beta_driving_traffic_percent -2.695289         0.246242      -10.945708   
 
    Robust p-value  
 0    0.000000e+00  
 1    0.000000e+00  
 2    6.982700e-06  
 3    0.000000e+00  


# Test the Null hypothesis that Model 2 is true model compared to Model 3.


In [17]:
results_model_2 = EstimationResults.from_yaml_file(filename='model_2.yaml')

model_2_AIC = results_model_2.akaike_information_criterion
model_3_AIC = results_model_3.akaike_information_criterion

if model_2_AIC > model_3_AIC :
    print('Model 3 is the true model')
else :
    print('Model 2 is the true model')


Model 3 is the true model


## 1. Box-Cox model (common lambda for all modes)

In [18]:
lambda_boxcox = Beta('lambda_boxcox', 0.5, None, None, 0)

v_walk_bc = beta_time_walk * models.boxcox(dur_walking, lambda_boxcox)

v_bike_bc = (asc_bike + beta_age * age
             + beta_time_bike * models.boxcox(dur_cycling, lambda_boxcox))

v_pt_bc = (asc_pt + beta_age * age + beta_cost * cost_pt_euro
           + beta_time_pt * models.boxcox(dur_pt_total, lambda_boxcox))

v_car_bc = (asc_car + beta_age * age + beta_cost * cost_car_euro
            + beta_time_car * models.boxcox(dur_driving, lambda_boxcox)
            + beta_driving_traffic_percent * driving_traffic_percent)

V_bc = {1: v_walk_bc, 2: v_bike_bc, 3: v_pt_bc, 4: v_car_bc}
logprob_bc = models.loglogit(V_bc, None, travel_mode)

biogeme_bc = bio.BIOGEME(database, logprob_bc)
biogeme_bc.model_name = 'model_boxcox'    # new name, so that no old result file is reused
results_bc = biogeme_bc.estimate()

In [19]:
print(results_bc.print_general_statistics())
boxcox_estimates = get_pandas_estimated_parameters(estimation_results=results_bc)["Estimated parameters"]
boxcox_estimates

Number of estimated parameters             11
Sample size                                5000
Excluded observations                      0
Init log likelihood                        -4260.533
Final log likelihood                       -4093.385
Likelihood ratio test for the init. model  334.2964
Rho-square for the init. model             0.0392
Rho-square-bar for the init. model         0.0366
Akaike Information Criterion               8208.77
Bayesian Information Criterion             8280.459
Final gradient norm                        2.9388E-03
Bootstrapping time                         None


,Name,Value,Robust std err.,Robust t-stat.,Robust p-value
0,beta_time_walk,-5.890210,0.304191,-19.363530,0.000000e+00
1,lambda_boxcox,0.420653,0.057699,7.290414,3.090861e-13
2,asc_bike,-2.755771,0.315090,-8.745991,0.000000e+00
3,beta_age,0.010513,0.002324,4.524658,6.049331e-06
4,beta_time_bike,-3.645556,0.322292,-11.311342,0.000000e+00
5,asc_pt,1.539087,0.199221,7.725527,1.110223e-14
6,beta_cost_car,-0.122654,0.015845,-7.740611,9.992007e-15
7,beta_time_pt,-2.585492,0.202935,-12.740493,0.000000e+00
8,asc_car,1.289901,0.303010,4.256956,2.072287e-05
9,beta_time_car,-2.991484,0.294177,-10.169002,0.000000e+00


### t-tests on $\lambda$

Under $H_0$, $(\hat\lambda - \lambda_0)/\hat\sigma_\lambda$ follows approximately a $N(0,1)$.

In [20]:
lam = boxcox_estimates.loc[boxcox_estimates["Name"] == 'lambda_boxcox', 'Value'].values[0]
se  = boxcox_estimates.loc[boxcox_estimates["Name"] == 'lambda_boxcox', 'Robust std err.'].values[0]

t_lambda_1 = (lam - 1) / se    # H0: lambda = 1 -> Model 2 is true
t_lambda_0 = (lam - 0) / se    # H0: lambda = 0 -> Model 3 is true

threshold_5 = norm.ppf(0.975)  # two-sided test at 5%

print(f'lambda_hat = {lam:.4f}  (std err = {se:.4f})')
print(f'H0: lambda = 1 (Model 2 true): t = {t_lambda_1:.3f}, p-value = {calc_p_value(t_lambda_1):.4f}, threshold = {threshold_5:.3f}')
print(f'H0: lambda = 0 (Model 3 true): t = {t_lambda_0:.3f}, p-value = {calc_p_value(t_lambda_0):.4f}, threshold = {threshold_5:.3f}')

lambda_hat = 0.4207  (std err = 0.0577)
H0: lambda = 1 (Model 2 true): t = -10.041, p-value = 0.0000, threshold = 1.960
H0: lambda = 0 (Model 3 true): t = 7.290, p-value = 0.0000, threshold = 1.960


### Likelihood ratio tests (1 degree of freedom)

In [21]:
lr_bc_vs_m2 = results_bc.likelihood_ratio_test(results_model_2, 0.05)   # H0: lambda = 1
lr_bc_vs_m3 = results_bc.likelihood_ratio_test(results_model_3, 0.05)   # H0: lambda = 0

print('H0: Model 2 is true (lambda = 1):', lr_bc_vs_m2, '| p-value =', chi2.sf(lr_bc_vs_m2.statistic, 1))
print('H0: Model 3 is true (lambda = 0):', lr_bc_vs_m3, '| p-value =', chi2.sf(lr_bc_vs_m3.statistic, 1))

H0: Model 2 is true (lambda = 1): H0 can be rejected at level 5.0% (statistic: 68.48, threshold: 3.84) | p-value = 1.2829880007398877e-16
H0: Model 3 is true (lambda = 0): H0 can be rejected at level 5.0% (statistic: 55.75, threshold: 3.84) | p-value = 8.22781613340877e-14


In [22]:
rej2 = abs(t_lambda_1) > threshold_5
rej3 = abs(t_lambda_0) > threshold_5

if rej2 and not rej3:
    print('Model 2 (lambda = 1) is rejected, Model 3 (lambda = 0) is not: Model 3 is the better model.')
elif rej3 and not rej2:
    print('Model 3 (lambda = 0) is rejected, Model 2 (lambda = 1) is not: Model 2 is the better model.')
else:
    best = 'Model 3' if abs(t_lambda_0) < abs(t_lambda_1) else 'Model 2'
    print(f'{best} is the better model: lambda_hat is closer to {"0" if best == "Model 3" else "1"} (smaller |t|).')

Model 3 is the better model: lambda_hat is closer to 0 (smaller |t|).
